In [1]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score 
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV

from diet_dataset import load_foods

In [2]:
food_df = load_foods('../data/food1.csv')
food_df

,Description,Data.Carbohydrate,Data.Cholesterol,keto,vegan
0,"BUTTER,WITH SALT",0.06,215,True,False
1,"BUTTER,WHIPPED,WITH SALT",0.06,219,True,False
2,"BUTTER OIL,ANHYDROUS",0.00,256,True,False
3,"CHEESE,BLUE",2.34,75,True,False
4,"CHEESE,BRICK",2.79,94,True,False
...,...,...,...,...,...
7429,EGG,0.00,100,True,False
7430,EGGS,0.00,100,True,False
7431,BUTTER,0.00,100,True,False
7432,"MILK,CASHEW",2.00,0,True,True


In [3]:
ground_truth_df = pd.read_csv('../data/ground_truth_sample.csv')
ground_truth_df

,title,description,instructions,ingredients,photo_url,vegan,keto
0,"Grandma's Eggless, Butterless, Milkless Cake",You can get a moist chocolate cake without egg...,"['In a large bowl, combine all the dry ingredi...",['3 cups all-purpose flour' '2 cups white suga...,http://images.media-allrecipes.com/userphotos/...,True,False
1,Sweet and Spicy Turkey Rub,This dry rub recipe is an adaptation of a caju...,"['Combine brown sugar, garlic powder, red pepp...",['3 tablespoons brown sugar' '2 tablespoons ga...,http://images.media-allrecipes.com/userphotos/...,True,False
2,Marinated Mushrooms with Red Bell Peppers,This is perfect for a buffet or as a football ...,"['Combine the vinegar, water, oil, sugar, onio...",['1/2 cup red wine vinegar' '1/3 cup water' '2...,http://images.media-allrecipes.com/userphotos/...,True,False
3,Easy 4-Ingredient Margarita,This margarita is so easy to make and is absol...,['Fill a cocktail shaker with ice; add tequila...,"['1 cup ice cubes, or as needed' '1/3 cup tequ...",http://images.media-allrecipes.com/userphotos/...,True,False
4,Two Bowl Cake,"I don't remember when I got this recipe, but i...",['Preheat oven to 350 degrees F (175 degrees C...,['3 cups all-purpose flour' '2 cups white suga...,http://images.media-allrecipes.com/userphotos/...,True,False
...,...,...,...,...,...,...,...
95,Gingery Lemonade,"Spiked with plenty of fresh ginger, this spice...",['In an 8-quart saucepan combine SPLENDA(R) Gr...,"['1 1/2 cups SPLENDA® No Calorie Sweetener, Gr...",http://images.media-allrecipes.com/userphotos/...,True,True
96,Simple Cajun Seasoning,Here is a simple way to make Cajun seasoning u...,"['Combine the salt, oregano, paprika, cayenne ...",['2 1/2 tablespoons salt' '1 tablespoon dried ...,http://images.media-allrecipes.com/userphotos/...,True,True
97,Lemon-Basil Vinaigrette Corn Topper,Summer corn gets even better with a squeeze of...,"['Whisk lemon juice, oil and basil in a small ...",['1 tablespoon lemon juice' '2 teaspoons extra...,http://images.media-allrecipes.com/global/reci...,True,True
98,Dry Spice Rub for Lamb or Beef,I got this recipe from my grama. It can be use...,"['Mix together the paprika, thyme, basil, cumi...",['1 teaspoon paprika' '1 1/2 teaspoons dried t...,http://images.media-allrecipes.com/userphotos/...,True,True


In [4]:
# training corupus
X_train = food_df['Description'].tolist()

# training labels - vegan
y_train_vegan = food_df.vegan
y_train_keto = food_df.keto

# ground truth
X_test = ground_truth_df['ingredients']
y_vegan_test = ground_truth_df['vegan']
y_keto_test = ground_truth_df['keto']

stop_words = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 
    'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 
    'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 
    'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves',
    'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are',
    'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing',
    'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 
    'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 
    'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 
    'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 
    'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 
    's', 't', 'can', 'will', 'just', 'don', "don't", 'should', "should've", 'now', 'd', 'll', 'm', 'o', 're', 've', 'y',
    'ain', 'aren', "aren't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 
    'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't", 'ma', 'mightn',
    "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 
    'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't",    
    "chopped", "large", "seeded", "drained", "rinsed", "peeled", "pitted", "ground", "processed", "sliced", "grated", 
    "cut", "chunks", "taste", "freshly", "fresh", "raw",
    'cups', 'cup', 
    'tablespoons', 'tablespoon', 
    'teaspoons', 'teaspoon',
    'can', 'cans',
    'packages', 'package',
    'ounce', 'ounces',
    'quart', 'quarts',
    'pound', 'pounds',
    'drop', 'drops',
    'pint', 'pints',
    'cube', 'cubes',
    'clove', 'cloves',
    'pinch', 'slices',
    'pinch' 
]

In [5]:
tvec_vegan = TfidfVectorizer(ngram_range=(1, 3), lowercase=True, stop_words=stop_words)
tvec_keto = TfidfVectorizer(ngram_range=(1, 3), lowercase=True, stop_words=stop_words)

### Logistic Regression

In [6]:
# Define the parameter grid
param_grid = {
    'logisticregression__C': [0.01, 0.1, 1, 10],
    'logisticregression__penalty': ['l2'],
    'logisticregression__solver': ['lbfgs', 'saga']
}

# Create pipelines
logreg_vegan_pipeline = make_pipeline(tvec_vegan, LogisticRegression(max_iter=1000))
logreg_keto_pipeline = make_pipeline(tvec_keto, LogisticRegression(max_iter=1000))

# Wrap with GridSearchCV
grid_vegan = GridSearchCV(logreg_vegan_pipeline, param_grid, cv=5, scoring='accuracy')
grid_keto = GridSearchCV(logreg_keto_pipeline, param_grid, cv=5, scoring='accuracy')

# Train the model
grid_vegan.fit(X_train, y_train_vegan)
grid_keto.fit(X_train, y_train_keto)

# Predictions
y_vegan_pred = grid_vegan.predict(X_test)
y_keto_pred = grid_keto.predict(X_test)

# Accuracy scores
print("Accuracy score for vegan/logreg" ,accuracy_score(y_vegan_test, y_vegan_pred))
print("Accuracy score for keto/logreg" ,accuracy_score(y_keto_test, y_keto_pred))

print("Best params for vegan:", grid_vegan.best_params_)
print("Best params for keto:", grid_keto.best_params_)

Accuracy score for vegan/logreg 0.79
Accuracy score for keto/logreg 0.58
Best params for vegan: {'logisticregression__C': 10, 'logisticregression__penalty': 'l2', 'logisticregression__solver': 'saga'}
Best params for keto: {'logisticregression__C': 10, 'logisticregression__penalty': 'l2', 'logisticregression__solver': 'lbfgs'}


Predict on new data

### Support Vector Classifier

In [7]:
svc_vegan_clf = make_pipeline(tvec_vegan, SVC())
svc_keto_clf = make_pipeline(tvec_keto, SVC())

# Train the model
svc_vegan_clf.fit(X_train, y_train_vegan)
svc_keto_clf.fit(X_train, y_train_keto)

# Train the model
y_vegan_pred = svc_vegan_clf.predict(X_test)
y_keto_pred = svc_keto_clf.predict(X_test)

# Accuracy scores
print("Accuracy score for vegan/svc" ,accuracy_score(y_vegan_test, y_vegan_pred))
print("Accuracy score for vegan/svc" ,accuracy_score(y_keto_test, y_keto_pred))

Accuracy score for vegan/svc 0.77
Accuracy score for vegan/svc 0.57


### Random Forest Classifier

In [8]:
rf_vegan_clf = make_pipeline(tvec_vegan, RandomForestClassifier())
rf_keto_clf = make_pipeline(tvec_keto, RandomForestClassifier())

# Train the models
rf_vegan_clf.fit(X_train, y_train_vegan)
rf_keto_clf.fit(X_train, y_train_keto)

# Predictions
y_vegan_pred = rf_vegan_clf.predict(X_test)
y_keto_pred = rf_keto_clf.predict(X_test)

# Accuracy scores
print("Accuracy score for vegan/rf:", accuracy_score(y_vegan_test, y_vegan_pred))
print("Accuracy score for keto/rf:", accuracy_score(y_keto_test, y_keto_pred))


Accuracy score for vegan/rf: 0.81
Accuracy score for keto/rf: 0.8


In [9]:
# Define the parameter grid for Random Forest
rf_param_grid = {
    'randomforestclassifier__n_estimators': [100, 200],
    'randomforestclassifier__max_depth': [None, 10, 20],
    'randomforestclassifier__min_samples_split': [2, 5],
    'randomforestclassifier__class_weight': [None, 'balanced']
}

# Create pipelines
rf_vegan_pipeline = make_pipeline(tvec_vegan, RandomForestClassifier(random_state=42))
rf_keto_pipeline = make_pipeline(tvec_keto, RandomForestClassifier(random_state=42))

# Wrap in GridSearchCV
grid_rf_vegan = GridSearchCV(rf_vegan_pipeline, rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf_keto = GridSearchCV(rf_keto_pipeline, rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit the models
grid_rf_vegan.fit(X_train, y_train_vegan)
grid_rf_keto.fit(X_train, y_train_keto)

# Predictions
y_vegan_pred = grid_rf_vegan.predict(X_test)
y_keto_pred = grid_rf_keto.predict(X_test)

# Results
print("Best params for vegan RF:", grid_rf_vegan.best_params_)
print("Accuracy score for vegan/rf:", accuracy_score(y_vegan_test, y_vegan_pred))

print("Best params for keto RF:", grid_rf_keto.best_params_)
print("Accuracy score for keto/rf:", accuracy_score(y_keto_test, y_keto_pred))


Best params for vegan RF: {'randomforestclassifier__class_weight': None, 'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_split': 5, 'randomforestclassifier__n_estimators': 100}
Accuracy score for vegan/rf: 0.8
Best params for keto RF: {'randomforestclassifier__class_weight': 'balanced', 'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_split': 5, 'randomforestclassifier__n_estimators': 100}
Accuracy score for keto/rf: 0.78


And we have our winners.
Next, I will serialize best models. Later, I will load it and use in my estimators.

In [12]:
import joblib

# Save the best model from RandomForestClassifier

joblib.dump(rf_vegan_clf, './vegan_clf.pkl')
joblib.dump(rf_vegan_clf, '../../web/src/vegan_clf.pkl')

joblib.dump(rf_keto_clf, './keto_clf.pkl')
joblib.dump(rf_keto_clf, '../../web/src/keto_clf.pkl')


['../../web/src/keto_clf.pkl']

Test loading serialized models

In [13]:
# Load the serialized models
loaded_rf_vegan = joblib.load('./vegan_clf.pkl')
loaded_rf_keto = joblib.load('./keto_clf.pkl')


X_loaded_test = [
    '3 eggs',
    '1 tablespoon butter',
    '3 cups all-purpose flour', 
    '2 cups white sugar'
    '6 tablespoons unsweetened cocoa powder', 
    '2 teaspoons baking soda',
    '2 teaspoons baking powder', 
    '2/3 cup vegetable oil', 
    '2 cups water',
    '2 tablespoons distilled white vinegar',
    '2 teaspoons vanilla extract',
    '1 tablespoon soy sauce' ,
    '2 garlic cloves, minced',
    '1 teaspoon sesame seeds, toasted' ,
    '1 teaspoon brown sugar',
    '1 teaspoon peanut butter' ,
    '3/4 pound fresh green beans, trimmed',
    '4 1/2 teaspoons vegetable oil',
 ]


y_vegan_pred = loaded_rf_vegan.predict(X_loaded_test)
y_keto_pred = loaded_rf_keto.predict(X_loaded_test)

df = pd.DataFrame({
    'igrideint': X_loaded_test,
    'vegan': y_vegan_pred,
    'keto': y_keto_pred
})
df

,igrideint,vegan,keto
0,3 eggs,False,True
1,1 tablespoon butter,False,True
2,3 cups all-purpose flour,True,False
3,2 cups white sugar6 tablespoons unsweetened co...,True,False
4,2 teaspoons baking soda,True,True
5,2 teaspoons baking powder,True,False
6,2/3 cup vegetable oil,True,True
7,2 cups water,True,True
8,2 tablespoons distilled white vinegar,True,True
9,2 teaspoons vanilla extract,True,False
